In [ ]:
#r "nuget: Azure.AI.Projects, 1.2.0-beta.5"
#r "nuget: Azure.Identity, 1.18.0-beta.2"
#r "nuget: Microsoft.Agents.AI, 1.0.0-rc1"
#r "nuget: Microsoft.Agents.AI.AzureAI, 1.0.0-rc1"

// Basic Agent Creation and Invocation sample using Azure Foundry Agents.

using Azure.AI.Projects;
using Azure.AI.Projects.OpenAI;
using Azure.Identity;
using Microsoft.Agents.AI;

var endpointUrl = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_ENDPOINT");
var endpoint = endpointUrl != null ? new Uri(endpointUrl) : throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_ENDPOINT environment variable is not set.");
var deploymentName = "gpt-4o";
var apiKey = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_API_KEY") ?? throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_API_KEY environment variable is not set.");
var githubToken = Environment.GetEnvironmentVariable("GITHUBTOKEN") ?? throw new InvalidOperationException("GITHUB_TOKEN environment variable is not set.");

const string agentName = "AgentFrameworkAgent-1";

var credOptions = new DefaultAzureCredentialOptions
{
TenantId = "<TenantId>" // Specify the desired tenant ID
};

// Get a client to create/retrieve/delete server side agents with Azure Foundry Agents.
AIProjectClient aiProjectClient = new AIProjectClient(new Uri(endpointUrl), new DefaultAzureCredential(credOptions));

AIAgent newAgent = await aiProjectClient.CreateAIAgentAsync(name: agentName, model: deploymentName, instructions: "You are very helpful assistant for providing information. Provide answer in brief");

// You can also get the AIAgent latest version by just providing its name.
AIAgent latestAgent = await aiProjectClient.GetAIAgentAsync(name: agentName);

// The AIAgent version can be accessed via the GetService method.
AgentVersion latestAgentVersion = latestAgent.GetService<AgentVersion>()!;
Console.WriteLine($"Latest agent version id: {latestAgentVersion.Id}");

// Once you have the AIAgent, you can invoke it like any other AIAgent.
Console.WriteLine(await latestAgent.RunAsync("Tell me about Microsoft Agent Framework."));



The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Azure.AI.Projects, 1.2.0-beta.5 Azure.Identity, 1.18.0-beta.2 Microsoft.Agents.AI, 1.0.0-rc1 Microsoft.Agents.AI.AzureAI, 1.0.0-rc1

Latest agent version id: AgentFrameworkAgent-1:20
Microsoft Agent was a technology introduced by Microsoft in 1997 to create interactive, animated characters. These characters could be integrated into applications to provide user interaction through speech, gestures, and text. The framework used animated characters like "Merlin" and "Clippy," allowing developers to incorporate speech recognition, text-to-speech, and scripting for creating dynamic user experiences.

Microsoft Agent became less relevant with advances in user interface design and was deprecated in Windows 7 and beyond, as newer technologies replaced it.



warning CS1702: Assuming assembly reference 'System.ClientModel, Version=1.8.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.Core' matches identity 'System.ClientModel, Version=1.8.1.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy



In [ ]:
#r "nuget: Azure.AI.Projects, 1.2.0-beta.5"
#r "nuget: Azure.Identity, 1.18.0-beta.2"
#r "nuget: Microsoft.Agents.AI, 1.0.0-rc1"
#r "nuget: Microsoft.Agents.AI.AzureAI, 1.0.0-rc1"

using Azure.AI.Projects;
using Azure.Identity;
using Microsoft.Agents.AI;

// Basic Agent Creation with streaming output.

var endpointUrl = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_ENDPOINT");
var endpoint = endpointUrl != null ? new Uri(endpointUrl) : throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_ENDPOINT environment variable is not set.");
var deploymentName = "gpt-4o";
var apiKey = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_API_KEY") ?? throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_API_KEY environment variable is not set.");
var githubToken = Environment.GetEnvironmentVariable("GITHUBTOKEN") ?? throw new InvalidOperationException("GITHUB_TOKEN environment variable is not set.");

const string agentName = "AgentFrameworkAgent-1";

var credOptions = new DefaultAzureCredentialOptions
{
TenantId = "<TenantId>" // Specify the desired tenant ID
};

// Get a client to create/retrieve/delete server side agents with Azure Foundry Agents.
AIProjectClient aiProjectClient = new AIProjectClient(new Uri(endpointUrl), new DefaultAzureCredential(credOptions));

// Define the agent you want to create. (Prompt Agent in this case)
AgentVersionCreationOptions options = new(new PromptAgentDefinition(model: deploymentName) { Instructions = "You are good at telling jokes." });

AIAgent newAgent = await aiProjectClient.CreateAIAgentAsync(name: agentName, model: deploymentName, instructions: "You are very helpful assistant for providing information. Provide answer in brief");

// You can also get the AIAgent latest version by just providing its name.
AIAgent latestAgent = await aiProjectClient.GetAIAgentAsync(name: agentName);

var updates = new List<AgentResponseUpdate>();

await foreach (AgentResponseUpdate update in latestAgent.RunStreamingAsync("How to make soup ?"))
{
    updates.Add(update);
    System.Console.Write(update);
}


Installed Packages Azure.AI.Projects, 1.2.0-beta.5 Azure.Identity, 1.18.0-beta.2 Microsoft.Agents.AI, 1.0.0-rc1 Microsoft.Agents.AI.AzureAI, 1.0.0-rc1

Making soup is simple! Here’s a basic guide:

1. **Choose a Base**: Start with broth (chicken, beef, vegetable) or water.

2. **Add Aromatics**: Sauté onions, garlic, and celery in oil or butter until soft.

3. **Include Vegetables**: Add carrots, potatoes, or any preferred veggies. Cook until slightly tender.

4. **Add Protein**: Incorporate chicken, beef, beans, or tofu if desired.

5. **Season**: Use salt, pepper, herbs like thyme or bay leaves. 

6. **Simmer**: Bring to a boil, then reduce heat to a simmer. Cook until all ingredients are tender.

7. **Finish**: Taste and adjust seasoning. Add greens (like spinach) in the last few minutes.

8. **Serve**: Enjoy hot with bread or crackers!

Feel free to customize based on preference!

In [ ]:
#r "nuget: Azure.AI.Projects, 1.2.0-beta.5"
#r "nuget: Azure.Identity, 1.18.0-beta.2"
#r "nuget: Microsoft.Agents.AI, 1.0.0-rc1"
#r "nuget: Microsoft.Agents.AI.AzureAI, 1.0.0-rc1"

using Azure.AI.Projects;
using Azure.Identity;
using Microsoft.Agents.AI;
using System;
using System.Threading;
using System.Threading.Tasks;
using System.Collections.Generic;
using Microsoft.Extensions.AI;

// Agent with Local Function Tools to get time.

// Simple Tools helper providing the methods referenced below.
public static class Tools
{
    public static Task<string> GetCurrentDateAndTime() =>
        Task.FromResult(DateTime.UtcNow.ToString("o"));

    public static Task<string> GetCurrentTimeZone() =>
        Task.FromResult(TimeZoneInfo.Local.StandardName);
}

var endpointUrl = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_ENDPOINT");
var endpoint = endpointUrl != null ? new Uri(endpointUrl) : throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_ENDPOINT environment variable is not set.");
var deploymentName = "gpt-4o";
var apiKey = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_API_KEY") ?? throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_API_KEY environment variable is not set.");
var githubToken = Environment.GetEnvironmentVariable("GITHUBTOKEN") ?? throw new InvalidOperationException("GITHUB_TOKEN environment variable is not set.");

const string agentName = "AgentFrameworkAgent-1";

var credOptions = new DefaultAzureCredentialOptions
{
TenantId = "<TenantId>" // Specify the desired tenant ID
};  

// Get a client to create/retrieve/delete server side agents with Azure Foundry Agents.
AIProjectClient aiProjectClient = new AIProjectClient(new Uri(endpointUrl), new DefaultAzureCredential(credOptions));

AIAgent timeAgent = await aiProjectClient.CreateAIAgentAsync(name: agentName, model: deploymentName, instructions: "You are a time expert", tools
: new List<AITool>()
{
    AIFunctionFactory.Create(Tools.GetCurrentDateAndTime, "current_date_and_time"),
    AIFunctionFactory.Create(Tools.GetCurrentTimeZone, "current_timezone")
});

AgentSession session = await timeAgent.CreateSessionAsync(CancellationToken.None);

Console.Write("Time Tool > ");
ChatMessage message = new ChatMessage(ChatRole.User, "What is the current date and time, and timezone?");
var timeResponse = await timeAgent.RunAsync(message, session);
Console.WriteLine(timeResponse);


Installed Packages Azure.AI.Projects, 1.2.0-beta.5 Azure.Identity, 1.18.0-beta.2 Microsoft.Agents.AI, 1.0.0-rc1 Microsoft.Agents.AI.AzureAI, 1.0.0-rc1

Time Tool > The current date and time is February 25, 2026, 06:03 AM (UTC). The timezone is Central Standard Time (CST).


In [ ]:
#r "nuget: Azure.AI.Projects, 1.2.0-beta.5"
#r "nuget: Azure.Identity, 1.18.0-beta.2"
#r "nuget: Microsoft.Agents.AI, 1.0.0-rc1"
#r "nuget: Microsoft.Agents.AI.AzureAI, 1.0.0-rc1"
#r "nuget: ModelContextProtocol, 0.4.1-preview.1"

using Azure.AI.Projects;
using Azure.Identity;
using Microsoft.Agents.AI;
using System;
using System.Threading;
using System.Threading.Tasks;
using System.Collections.Generic;
using ModelContextProtocol.Client;
using Microsoft.Extensions.AI;

var endpointUrl = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_ENDPOINT");
var endpoint = endpointUrl != null ? new Uri(endpointUrl) : throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_ENDPOINT environment variable is not set.");
var deploymentName = "gpt-4o";
var apiKey = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_API_KEY") ?? throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_API_KEY environment variable is not set.");
var githubToken = Environment.GetEnvironmentVariable("GITHUBTOKEN") ?? throw new InvalidOperationException("GITHUB_TOKEN environment variable is not set.");

const string agentName = "AgentFrameworkAgent-1";

var credOptions = new DefaultAzureCredentialOptions
{
TenantId = "<TenantId>" // Specify the desired tenant ID
};

// Get a client to create/retrieve/delete server side agents with Azure Foundry Agents.
AIProjectClient aiProjectClient = new AIProjectClient(new Uri(endpointUrl), new DefaultAzureCredential(credOptions));

McpClient mcpClient = await McpClient.CreateAsync(new HttpClientTransport(new HttpClientTransportOptions
{
   TransportMode = HttpTransportMode.StreamableHttp,
   Endpoint = new Uri("https://api.githubcopilot.com/mcp"),
   AdditionalHeaders = new Dictionary<string, string>()
     {
         {  "Authorization", githubToken }
     }
}));

IList<McpClientTool> tools = await mcpClient.ListToolsAsync();

Console.WriteLine($"Tools Count: {tools.Count}");
 

AIAgent mcpAgent = await aiProjectClient.CreateAIAgentAsync(name: agentName, model: deploymentName, instructions: "You are a github expert.", tools: tools.Cast<AITool>().ToList());

AgentSession mcpAgentSession = await mcpAgent.CreateSessionAsync(CancellationToken.None);
Console.WriteLine("MCP Tool > Find number of open issues in microsoft/agent-framework repo");
ChatMessage mcpMessage = new ChatMessage(ChatRole.User, "Get the number of open issues in microsoft/agent-framework repo");
var mcpResponse = await mcpAgent.RunAsync(mcpMessage, mcpAgentSession);
Console.WriteLine(mcpResponse);


Installed Packages Azure.AI.Projects, 1.2.0-beta.5 Azure.Identity, 1.18.0-beta.2 Microsoft.Agents.AI, 1.0.0-rc1 Microsoft.Agents.AI.AzureAI, 1.0.0-rc1 ModelContextProtocol, 0.4.1-preview.1

Tools Count: 43
MCP Tool > Find number of open issues in microsoft/agent-framework repo
The repository `microsoft/agent-framework` currently has 577 open issues.
